# j-lens

Minimal implementation of the Jacobian lens and J-space methods from
[*Verbalizable Representations Form a Global Workspace in Language Models*](https://transformer-circuits.pub/2026/workspace/index.html) (Gurnee et al., 2026), on Qwen3.5-4B.

The lens transports a residual-stream activation $h_\ell$ into the final-layer basis with the corpus-averaged Jacobian $J_\ell = \mathbb{E}\left[\partial h_{\text{final}} / \partial h_\ell\right]$ and decodes it with the model's own unembedding: $\text{lens}(h_\ell) = \mathrm{softmax}(W_U\, \mathrm{norm}(J_\ell h_\ell))$.

In [ ]:
import os
import torch

import evals
import jlens
import interventions as iv

MODEL = "Qwen/Qwen3.5-4B"  # or "Qwen/Qwen3.5-4B-Base"
LENS_PATH = "lens.pt"
N_FIT_PROMPTS = 100  # paper: 10 already beats baselines, released lenses use 1000

## 1. Model and lens

Fit $J_\ell$ on WikiText-103 (one forward + $\lceil d_{\text{model}} / \text{dim\_batch} \rceil$ backward passes per prompt), or load a previous fit. Set `USE_RELEASED = True` to use the official pre-fitted lens from `neuronpedia/jacobian-lens` instead.

In [ ]:
model, tok = jlens.load_model(MODEL)

USE_RELEASED = False
if USE_RELEASED:
    lens = jlens.JLens.from_pretrained(
        "neuronpedia/jacobian-lens",
        "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
        revision="qwen-n1000")
elif os.path.exists(LENS_PATH):
    lens = jlens.JLens.load(LENS_PATH)
else:
    prompts = jlens.load_wikitext(N_FIT_PROMPTS)
    lens = jlens.fit(model, tok, prompts, dim_batch=8, checkpoint="fit_ckpt.pt")
    lens.save(LENS_PATH)
lens

## 2. Readout: J-lens vs logit lens

A two-hop prompt, read at the last token across layers. The J-lens should surface the unspoken intermediate (*Italy*) at depths where the logit lens is still noise (paper §2.4, §A.5).

In [ ]:
def show(prompt, layers, position=-1):
    jl, _, _ = lens.readout(model, tok, prompt, layers=layers, positions=[position])
    ll, ml, _ = lens.readout(model, tok, prompt, layers=layers, positions=[position],
                             use_jacobian=False)
    top = lambda lg: [tok.decode([t]) for t in lg.topk(5).indices]
    for l in reversed(layers):
        print(f"L{l:>2}  J-lens: {top(jl[l][0])!r:<60} logit-lens: {top(ll[l][0])!r}")
    print(f"model output: {top(ml[0])!r}")

n = len(model.model.layers)
layers = sorted({round(x * (n - 2)) for x in (0.2, 0.35, 0.5, 0.65, 0.8, 0.95)})
show("Fact: The currency used in the country shaped like a boot is the", layers)

In [ ]:
show("calc: (4 + 17) * 2 + 7 =", layers)  # intermediates 21, 42 en route to 49 (paper Fig 17)

## 3. Workspace band

Three cheap signatures from §4.1, per layer over held-out WikiText: excess kurtosis of the lens readout, top-10 agreement with the model's next token, and top-1 autocorrelation across positions vs a shuffled null. The workspace is the band where kurtosis and autocorrelation are high but next-token agreement has not yet spiked ("motor" layers).

In [ ]:
import matplotlib.pyplot as plt

m = evals.band_metrics(model, tok, lens, jlens.load_wikitext(120)[100:])
fig, axes = plt.subplots(1, 3, figsize=(12, 2.8))
for ax, key in zip(axes, ["kurtosis", "next_token_agree", "autocorr"]):
    ax.plot(list(m), [m[l][key] for l in m])
    ax.set_title(key); ax.set_xlabel("layer")
plt.tight_layout()

In [ ]:
WORKSPACE = [l for l in range(round(0.38 * n), round(0.92 * n))]  # adjust from the curves above
WORKSPACE

## 4. pass@k evaluation

The six prompt sets of §A.6 (downloaded from the companion repo). An intermediate is recovered at $k$ if its min-over-layers lens rank is $\le k$; the summary number is the AUC of pass@$k$ against $\log k$, normalized so always-rank-1 scores 1. The J-lens should beat the logit lens on every set, most on multilingual / order-ops / poetry / typo.

In [ ]:
for name in evals.EVALS:
    _, auc_j = evals.pass_at_k(model, tok, lens, name)
    _, auc_l = evals.pass_at_k(model, tok, lens, name, use_jacobian=False)
    print(f"{name:<14} J-lens {auc_j:.3f}   logit-lens {auc_l:.3f}")

items = evals.fetch("experiments", "probe-swap")["items"]
n_base = n_swap = n_swap_on_correct = used = 0
for item in items:
    pairs = evals.pair_token_ids(tok, item["intermediate"], item["swap_to"])
    ans = evals.token_ids_of(tok, item["answer"])
    swap_ans = evals.token_ids_of(tok, item["swap_answer"])
    if not (pairs and ans and swap_ans):
        continue
    used += 1
    prompt = item["prompt"].rstrip()
    base_ok = iv.logits_with(model, tok, prompt).argmax().item() in ans
    edits = iv.swap_edits(lens, model, WORKSPACE, pairs, alpha=1.0)
    swap_ok = iv.logits_with(model, tok, prompt, edits).argmax().item() in swap_ans
    n_base += base_ok
    n_swap += swap_ok
    n_swap_on_correct += base_ok and swap_ok
print(f"baseline correct {n_base}/{used}   swap redirects {n_swap}/{used}"
      f"   swap on baseline-correct {n_swap_on_correct}/{n_base}")
# paper (single-token multihop, alpha=1): 54% on Haiku 4.5, 70% on Sonnet 4.5

In [ ]:
data = evals.fetch("experiments", "verbal-report")["candidates"]
chat = lambda cat: tok.apply_chat_template(
    [{"role": "user", "content": f"Think of a {cat}. Answer in one word."}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False)
top = lambda lg: [(tok.decode([t]), round(lg[t].item() - lg.max().item(), 2)) for t in lg.topk(5).indices]

n_ok = n_tot = 0
for cat, words in data.items():
    prompt = chat(cat)
    clean = iv.logits_with(model, tok, prompt)
    answer = tok.decode([clean.argmax()]).strip()
    for cand in [w for w in words if w.lower() != answer.lower()][:5]:
        pairs = evals.pair_token_ids(tok, answer, cand)
        cand_ids = evals.token_ids_of(tok, cand)
        if not (pairs and cand_ids):
            continue
        n_tot += 1
        sw = iv.logits_with(model, tok, prompt, iv.swap_edits(lens, model, WORKSPACE, pairs))
        n_ok += sw.argmax().item() in cand_ids
print(f"verbal-report swaps: {n_ok}/{n_tot} report the swapped-in item")

prompt = chat("sport")
clean = iv.logits_with(model, tok, prompt)
answer = tok.decode([clean.argmax()]).strip()
target = "Rugby" if answer.lower() != "rugby" else "Tennis"
sw = iv.logits_with(model, tok, prompt, iv.swap_edits(lens, model, WORKSPACE, evals.pair_token_ids(tok, answer, target)))
print("clean:", top(clean))
print(f"swap {answer} -> {target}:", top(sw))

In [ ]:
items = evals.fetch("experiments", "probe-swap")["items"]
ok_base = ok_swap = used = 0
for item in items[:30]:
    src = evals.token_ids_of(tok, item["intermediate"])
    tgt = evals.token_ids_of(tok, item["swap_to"])
    ans = evals.token_ids_of(tok, item["answer"])
    swap_ans = evals.token_ids_of(tok, item["swap_answer"])
    if not (src and tgt and ans and swap_ans):
        continue
    used += 1
    ok_base += iv.logits_with(model, tok, item["prompt"]).argmax().item() in ans
    edits = iv.swap_edits(lens, model, WORKSPACE, [(src[0], tgt[0])], alpha=1.0)
    ok_swap += iv.logits_with(model, tok, item["prompt"], edits).argmax().item() in swap_ans
print(f"baseline correct {ok_base}/{used}   swap redirects answer {ok_swap}/{used}")
# paper (single-token multihop set): 54% on Haiku 4.5, 70% on Sonnet 4.5

## 6. J-space decomposition

Sparse nonnegative pursuit of an activation against the layer's lens vectors (§2.3): the few atoms name what the model is holding at that position; the residual is the non-J-space component (which the paper finds carries most of the variance but little of the causal effect on report/reasoning).

In [ ]:
prompt = "Fact: The number of legs on the animal that spins webs is "
layer = WORKSPACE[len(WORKSPACE) // 2]
with jlens.record_residuals(model, [layer]) as rec, torch.no_grad():
    model.model(input_ids=jlens.encode(model, tok, prompt), use_cache=False)
h = rec.acts[layer][0, -1]

ids, coefs, recon = lens.decompose(model, layer, h, k=16)
print([(tok.decode([t]), round(c.item(), 2)) for t, c in zip(ids, coefs)])
print(f"J-space component: {(recon.norm() / h.float().cpu().norm()).item():.1%} of activation norm")